


## Install PySpark

In [1]:
!pip install -q pyspark

## Data Lake setup


In [2]:
import os, shutil, hashlib
from google.colab import files

BRONZE, SILVER = "data_lake/bronze", "data_lake/silver"
OUT = f"{SILVER}/preprocessed_sales"
os.makedirs(BRONZE, exist_ok=True); os.makedirs(SILVER, exist_ok=True)

if not os.path.exists(f"{BRONZE}/sales.csv"):
    if os.path.exists("sales.csv"):
        shutil.copy("sales.csv", f"{BRONZE}/sales.csv")
    else:
        up = files.upload()
        shutil.copy(list(up.keys())[0], f"{BRONZE}/sales.csv")

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
BRONZE_MD5 = md5(f"{BRONZE}/sales.csv")
print(os.listdir(BRONZE), os.listdir(SILVER))

['sales.csv'] ['sales_silver.csv', 'preprocessed_sales']


## Load and inspect with Spark

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("local[*]").appName("SalesDataLake")
         .config("spark.sql.ansi.enabled", "false").getOrCreate())

raw = spark.read.option("header", True).option("inferSchema", False).csv(f"{BRONZE}/sales.csv")
raw.printSchema()
raw.show(5, truncate=False)
RAW_COUNT = raw.count()
print("Total records (Bronze):", RAW_COUNT)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+---

## Detect data-quality issues

In [4]:
ID_COLS   = ["order_id", "customer_id"]
TEXT_COLS = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
cols = raw.columns

NULL_TOKENS = ["", "null", "none", "nan", "n/a", "na", "-", "?"]
is_missing = lambda c: F.col(c).isNull() | F.lower(F.trim(F.col(c))).isin(NULL_TOKENS)
to_num = lambda c: F.expr(f"try_cast(regexp_replace(trim(`{c}`), '[^0-9.-]', '') as double)")

DATE_FORMATS = ["yyyy-M-d", "yyyy/M/d", "d/M/yyyy", "d-M-yyyy", "d.M.yyyy", "M/d/yyyy", "M-d-yyyy",
                "d-MMM-yyyy", "d MMM yyyy", "MMM d, yyyy", "MMMM d, yyyy"]   # day-first for ambiguous dates
def parse_date(c):
    d = F.coalesce(*[F.to_date(F.trim(F.col(c)), f) for f in DATE_FORMATS])
    return F.when(d.between(F.lit("1990-01-01").cast("date"), F.current_date()), d)  # out of range -> null

#Duplicates
print("Exact duplicate rows:", RAW_COUNT - raw.distinct().count())

#Missing values per column
raw.select([F.sum(is_missing(c).cast("int")).alias(c) for c in cols]).show()

#Text inconsistencies (spaces / case variants)
for c in TEXT_COLS:
    s = F.col(c)
    r = raw.filter(~is_missing(c)).agg(
        F.sum((s != F.trim(s)).cast("int")).alias("extra_spaces"),
        F.countDistinct(s).alias("distinct_raw"),
        F.countDistinct(F.lower(F.trim(s))).alias("distinct_normalised")).collect()[0]
    print(c, r.asDict())

#Invalid numbers
raw.select(
    F.sum((to_num("quantity") <= 0).cast("int")).alias("qty_<=0"),
    F.sum((to_num("unit_price") <= 0).cast("int")).alias("price_<=0"),
    F.sum(((to_num("discount_percent") < 0) | (to_num("discount_percent") > 100)).cast("int")).alias("disc_outside_0_100"),
).show()

#Invalid / inconsistent dates
bad = raw.filter(~is_missing("order_date") & parse_date("order_date").isNull())
print("Invalid dates:", bad.count()); bad.select("order_id", "order_date").show()
(raw.withColumn("pattern", F.regexp_replace(F.regexp_replace("order_date", "[0-9]", "9"), "[A-Za-z]", "a"))
    .groupBy("pattern").count().show())

Exact duplicate rows: 10
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

customer_name {'extra_spaces': 3, 'distinct_raw': 305, 'distinct_normalised': 302}
product {'extra_spaces': 0, 'distinct_raw': 57, 'distinct_normalised': 57}
category {'extra_spaces': 0, 'distinct_raw': 13, 'distinct_normalised': 8}
payment_method {'extra_spaces': 0, '

## Preprocessing

In [5]:
log = {}

#Remove exact duplicates
df = raw.dropDuplicates()
log["Exact duplicate rows removed"] = RAW_COUNT - df.count()

#Trim / collapse spaces, placeholders -> null
for c in cols:
    t = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    df = df.withColumn(c, F.when(F.lower(t).isin(NULL_TOKENS), None).otherwise(t))

#Standardise capitalisation (Title Case) for columns that had case variants
#    (product and payment_method are only trimmed, so "UPI" / "USB-C Hub" stay intact)
CASE_COLS = ["customer_name", "category", "city", "state", "order_status"]
changed = 0
for c in CASE_COLS:
    new = F.initcap(F.lower(F.col(c)))
    changed += df.filter(F.col(c).isNotNull() & (F.col(c) != new)).count()
    df = df.withColumn(c, new)
log["Text cells re-capitalised"] = changed
n = df.count(); df = df.dropDuplicates()
log["Extra duplicates removed after standardising"] = n - df.count()

#Validate quantity (positive integer), unit price (>0), discount (0-100) -> invalid = null
q, p, d = to_num("quantity"), to_num("unit_price"), to_num("discount_percent")
log["Invalid quantity -> null"]   = df.filter(~(q > 0) & F.col("quantity").isNotNull()).count()
log["Invalid unit_price -> null"] = df.filter(~(p > 0) & F.col("unit_price").isNotNull()).count()
log["Invalid discount -> null"]   = df.filter(~d.between(0, 100) & F.col("discount_percent").isNotNull()).count()
df = (df.withColumn("quantity", F.when((q > 0) & (q == F.floor(q)), q.cast("int")))
        .withColumn("unit_price", F.when(p > 0, F.round(p, 2)))
        .withColumn("discount_percent", F.when(d.between(0, 100), F.round(d, 2))))

#Dates -> yyyy-MM-dd, invalid -> null
dt = parse_date("order_date")
log["Invalid dates -> null"] = df.filter(F.col("order_date").isNotNull() & dt.isNull()).count()
df = df.withColumn("order_date", F.date_format(dt, "yyyy-MM-dd"))

#Missing values
med_q = df.approxQuantile("quantity", [0.5], 0.001)[0]
med_p = df.approxQuantile("unit_price", [0.5], 0.001)[0]
log[f"quantity nulls filled with median ({med_q})"]   = df.filter(F.col("quantity").isNull()).count()
log[f"unit_price nulls filled with median ({med_p})"] = df.filter(F.col("unit_price").isNull()).count()
log["discount_percent nulls filled with 0"]           = df.filter(F.col("discount_percent").isNull()).count()
log["text nulls filled with 'Unknown'"] = sum(df.filter(F.col(c).isNull()).count() for c in TEXT_COLS)

df = (df.fillna({"quantity": int(med_q), "unit_price": med_p, "discount_percent": 0.0})
        .fillna("Unknown", subset=TEXT_COLS))

for k, v in log.items(): print(f"{k:55s} {v}")

Exact duplicate rows removed                            10
Text cells re-capitalised                               12
Extra duplicates removed after standardising            0
Invalid quantity -> null                                12
Invalid unit_price -> null                              8
Invalid discount -> null                                8
Invalid dates -> null                                   4
quantity nulls filled with median (5.0)                 12
unit_price nulls filled with median (1840.0)            8
discount_percent nulls filled with 0                    8
text nulls filled with 'Unknown'                        26


## Save to Silver

In [6]:
SILVER_COUNT = df.count()
df.coalesce(1).write.mode("overwrite").option("header", True).csv(OUT)

part = [f for f in os.listdir(OUT) if f.startswith("part-") and f.endswith(".csv")][0]
shutil.copy(f"{OUT}/{part}", f"{SILVER}/sales_silver.csv")
print(os.listdir(OUT)); print(os.listdir(SILVER))

['.part-00000-f91b3ae6-00ab-46db-8f3f-8c7fbb2dfa23-c000.csv.crc', '._SUCCESS.crc', '_SUCCESS', 'part-00000-f91b3ae6-00ab-46db-8f3f-8c7fbb2dfa23-c000.csv']
['sales_silver.csv', 'preprocessed_sales']


## Verification

In [7]:
print("Rows before (Bronze):", RAW_COUNT)
print("Rows after  (Silver):", SILVER_COUNT)
assert md5(f"{BRONZE}/sales.csv") == BRONZE_MD5; print("Bronze unchanged ✔")

silver = spark.read.option("header", True).option("inferSchema", True).csv(OUT)
silver.printSchema(); silver.show(10, truncate=False)
print("Rows read back:", silver.count())
silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver.columns]).show()

print("\nPreprocessing operations performed:")
for k, v in log.items(): print(f"- {k}: {v}")

Rows before (Bronze): 1000
Rows after  (Silver): 990
Bronze unchanged ✔
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+----------------------+-----------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name |product               |category   |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|
+--------+-----------+--------

## Summary of preprocessing
- Removed 10 exact duplicate rows (1000 → 990 rows).
- Trimmed extra spaces and standardized capitalization in name, category, city, state and status.
- Set invalid quantity (12), unit price (8) and discount (8) values to null, then filled them (median for quantity and price, 0 for discount).
- Converted valid dates to yyyy-MM-dd and set 4 invalid dates to null.
- Filled 26 missing text values with "Unknown".
- Saved the result to data_lake/silver/preprocessed_sales/ and read it back with Spark.
